In [ ]:
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from src.logger import logging
import mlflow
import dagshub
from sklearn.model_selection import GridSearchCV

In [2]:
df = pd.read_csv(r'D:\Resume-Screening-Matching-System\data_upload_S3\data\cleaned_dataset.csv')
df.head()

,Role,Resume
0,E-commerce Specialist,Here's a professional resume for Jason Jones:\...
1,Game Developer,Here's a professional resume for Ann Marshall:...
2,Human Resources Specialist,Here's a professional resume for Patrick Mccla...
3,E-commerce Specialist,Here's a professional resume for Patricia Gray...
4,E-commerce Specialist,Here's a professional resume for Amanda Gross:...


Preprocessing Data:

In [3]:
def clean_text(text):
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)

    # Remove URLs (linkedin, github, websites)
    text = re.sub(r'http\S+|www\.\S+|linkedin\.com/\S+|github\.com/\S+', ' ', text)

    # Remove phone numbers
    text = re.sub(
        r'(\+\d{1,3}[-.\s]?)?(\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4})',
        ' ',
        text
    )

    # Remove markdown links
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)

    # Remove bullet symbols
    text = re.sub(r'[•▪◦●■♦★]', ' ', text)

    # Remove asterisks used as bullets
    text = re.sub(r'\*', ' ', text)

    # Keep only letters, numbers, +, #, /, . and spaces
    text = re.sub(r'[^a-zA-Z0-9+#/. ]', ' ', text)

    # Convert to lowercase
    text = text.lower()

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    

    return text

def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def normalize_text(text):
    """Normalize the text by applying all preprocessing steps."""
    text = clean_text(text)
    text = lemmatization(text)
    text = remove_stop_words(text)
    return text

In [4]:
df['Resume'] = df['Resume'].apply(lambda x: normalize_text(x))

In [5]:
df['Resume'][0]

'professional resume jason jones jason jones e commerce specialist contact information email phone linkedin summary result driven e commerce specialist 5+ year experience inventory management seo online advertising analytics. proven track record increasing online sale improving website traffic optimizing inventory levels. skilled analyzing complex data set identifying trend making data driven decisions. passionate staying date latest e commerce trend technologies. professional experience e commerce specialist xyz corporation 2018 present managed inventory level across multiple channel resulting 25 reduction stockouts 15 reduction overstocking developed implemented seo strategy increased website traffic 30 improved search engine ranking 20 created executed online advertising campaign generated 50 increase sale 20 increase conversion rate analyzed website analytics identify trend optimize user experience improve customer engagement collaborated cross functional team launch new product li

Target Column into Categorical values

In [6]:
label = LabelEncoder()
df['Role'] = label.fit_transform(df['Role'])
df.head()

,Role,Resume
0,15,professional resume jason jones jason jones e ...
1,17,professional resume ann marshall ann marshall ...
2,18,professional resume patrick mcclain patrick mc...
3,15,professional resume patricia gray patricia gra...
4,15,professional resume amanda gross amanda gross ...


Train Test Split:

In [7]:
X_train, X_test, y_train, y_test = train_test_split(df['Resume'], df['Role'], test_size=0.2, random_state=42)

In [8]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((8139,), (2035,), (8139,), (2035,))

Vectorizer

In [9]:
vectorizer = TfidfVectorizer(
    max_features=200,
    ngram_range=(1,2)
)
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

In [ ]:
CONFIG = {
    "test_size": 0.2,
    "mlflow_tracking_uri": "https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow",
    "dagshub_repo_owner": "mdzaid382",
    "dagshub_repo_name": "Resume-Screening-Matching-System",
    "experiment_name": "svc_tf-idf"
}

mlflow.set_tracking_uri(CONFIG["mlflow_tracking_uri"])
dagshub.init(repo_owner=CONFIG["dagshub_repo_owner"], repo_name=CONFIG["dagshub_repo_name"], mlflow=True)
mlflow.set_experiment(CONFIG["experiment_name"])

[ 2026-06-03 16:15:06,020 ] httpx - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/mdzaid382/Resume-Screening-Matching-System "HTTP/1.1 200 OK"


Initialized MLflow to track repo "mdzaid382/Resume-Screening-Matching-System"

[ 2026-06-03 16:15:06,025 ] dagshub - INFO - Initialized MLflow to track repo "mdzaid382/Resume-Screening-Matching-System"


Repository mdzaid382/Resume-Screening-Matching-System initialized!

[ 2026-06-03 16:15:06,027 ] dagshub - INFO - Repository mdzaid382/Resume-Screening-Matching-System initialized!


<Experiment: artifact_location='mlflow-artifacts:/b1c65908c36646e48a57003cc86ae250', creation_time=1780482986782, experiment_id='4', last_update_time=1780482986782, lifecycle_stage='active', name='svc_tfidf', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [16]:
param_grid = {
        "C": [0.1, 1, 10],
        "penalty": ["l1", "l2"],
    }
    
with mlflow.start_run():
    grid_search = GridSearchCV(LinearSVC(), param_grid, cv=5, scoring="f1_weighted", n_jobs=-1)
    grid_search.fit(X_train, y_train)
    # Log all hyperparameter tuning runs
    for params, mean_score, std_score in zip(grid_search.cv_results_["params"], 
                                                 grid_search.cv_results_["mean_test_score"], 
                                                 grid_search.cv_results_["std_test_score"]):
        with mlflow.start_run(run_name=f"SVC with params: {params}", nested=True):
            model = LinearSVC(**params)
            model.fit(X_train, y_train)
        
            y_pred = model.predict(X_test)
        
            metrics = {
                "accuracy": accuracy_score(y_test, y_pred),
                "precision": precision_score(y_test, y_pred, average='weighted'),
                "recall": recall_score(y_test, y_pred, average='weighted'),
                "f1_score": f1_score(y_test, y_pred, average='weighted'),
                "mean_cv_score": mean_score,
                "std_cv_score": std_score
            }
            # Log parameters & metric
            mlflow.log_params(params)
            mlflow.log_metrics(metrics)
            
            print(f"Params: {params} | Accuracy: {metrics['accuracy']:.4f} | F1: {metrics['f1_score']:.4f}")

        # Log the best model
    best_params = grid_search.best_params_
    best_model = grid_search.best_estimator_
    best_f1 = grid_search.best_score_
    mlflow.log_params(best_params)
    mlflow.log_metric("best_f1_score", best_f1)
    mlflow.sklearn.log_model(best_model, "model")
    
    print(f"\nBest Params: {best_params} | Best F1 Score: {best_f1:.4f}")

d:\Resume-Screening-Matching-System\venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Params: {'C': 0.1, 'penalty': 'l1'} | Accuracy: 0.9749 | F1: 0.9748
🏃 View run SVC with params: {'C': 0.1, 'penalty': 'l1'} at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/4/runs/a36939cb3a8043f3873948972c2dcb3b
🧪 View experiment at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/4
Params: {'C': 0.1, 'penalty': 'l2'} | Accuracy: 0.9887 | F1: 0.9887
🏃 View run SVC with params: {'C': 0.1, 'penalty': 'l2'} at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/4/runs/4c354376298845ff988288422dc7db56
🧪 View experiment at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/4


d:\Resume-Screening-Matching-System\venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Params: {'C': 1, 'penalty': 'l1'} | Accuracy: 0.9921 | F1: 0.9921
🏃 View run SVC with params: {'C': 1, 'penalty': 'l1'} at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/4/runs/4fe52a8d0b5c4c6e8a0d653028e0bae5
🧪 View experiment at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/4
Params: {'C': 1, 'penalty': 'l2'} | Accuracy: 0.9946 | F1: 0.9946
🏃 View run SVC with params: {'C': 1, 'penalty': 'l2'} at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/4/runs/8ba239190bf7489c95a923507d562c06
🧪 View experiment at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/4


d:\Resume-Screening-Matching-System\venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Params: {'C': 10, 'penalty': 'l1'} | Accuracy: 0.9931 | F1: 0.9931
🏃 View run SVC with params: {'C': 10, 'penalty': 'l1'} at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/4/runs/4e6af75b7b5648d48fa8b92b3349556f
🧪 View experiment at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/4
Params: {'C': 10, 'penalty': 'l2'} | Accuracy: 0.9961 | F1: 0.9961
🏃 View run SVC with params: {'C': 10, 'penalty': 'l2'} at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/4/runs/162f6f121c09435cb9f9d7a4c49a467a
🧪 View experiment at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/4


2026/06/03 16:17:35 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



Best Params: {'C': 10, 'penalty': 'l2'} | Best F1 Score: 0.9948
🏃 View run nimble-lynx-266 at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/4/runs/7f21b036f4074d4fb153b74cffa084ea
🧪 View experiment at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/4
